#### get text content from data

In [1]:
import pandas as pd
import re
import json
import ast
from os import fdopen, remove
from tempfile import mkstemp
from shutil import move, copymode
from pprint import pprint
import os
import string
from importlib.resources import path
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from nltk.translate.bleu_score import sentence_bleu

def flatten(xss):
    return [x for xs in xss for x in xs]

In [3]:
# preprocessing auxiliary functions

stopwords = open("config_files/spacy_stopwords.txt").read().replace("{", '[').replace("}",']')
stopwords = ast.literal_eval(stopwords)

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out


def remove_stop_words(token_list):
    return [token for token in token_list if not token.is_stop]

    
def lemmatizing(token_list):
    return [token.lemma_ for token in token_list]
    
def get_POS(token_list):
    return [token.pos_ for token in token_list]

In [4]:
# AI-terms from ANTHROSCORE paper, transalted with deepL
ai_terms = list(ast.literal_eval(open("config_files/ger_anthroscore_KI_terms.txt").read()).values())
regex_ai = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in ai_terms])[:-1]

# get AI-related words, sorted according to 'Häufigkeitsklassen'
# https://corpora.uni-leipzig.de/de/res?corpusId=deu_news_2024&word=KI
ki_terms = ['KI','Künstliche Intelligenz', 'ChatGPT', 'AI', 'Copilot', 'Gemini', 'Chatbot','LLM', 'Chat-GPT',
            'Sprachmodell', 'KI-Modell', 'KI-System', 'KI-Tool', 'Artificial Intelligence', 'KI-Anwendung'] + ai_terms
ki_terms = [x.lower() for x in ki_terms]
regex_ki = "".join(["\\b"+x.replace(" ","\\s")+"\\b|" for x in ki_terms])[:-1]
print(regex_ki)

\bki\b|\bkünstliche\sintelligenz\b|\bchatgpt\b|\bai\b|\bcopilot\b|\bgemini\b|\bchatbot\b|\bllm\b|\bchat-gpt\b|\bsprachmodell\b|\bki-modell\b|\bki-system\b|\bki-tool\b|\bartificial\sintelligence\b|\bki-anwendung\b|\bpalm\b|\blms\b|\bllama\b|\btransformer\b|\bsprachmodelle\b|\bsprachmodell\b|\bgpt\b|\bplms\b|\bvortrainierte\ssprachmodelle\b|\bgpt-2\b|\bxlnet\b|\bgroße\ssprachmodelle\b|\bllms\b|\bgpt-3\b|\bstiftungsmodell\b|\bgpt-neo\b|\bgpt-j\b|\bchatgpt\b|\bgpt-4\b


In [5]:
# get sentences with ki_terms
def to_sentences(sentences_file):
    sentences = []
    with open(sentences_file) as f:
        for line in f:
            txt = line.split('\t')
            #split sentences
            txt_content = txt[1::]
            # filter ki_terms
            sentences.append(remove_punctuation(line.lower()))  
    return sentences


leipzig_main = "deu_news"
ki_data = {}
sentences_all = {}
for p in os.listdir(leipzig_main):
    k = p.replace("deu_news_","")
    sentences_file = os.path.join(leipzig_main, p, f"{p}-sentences.txt")
    # extract ki_sentences
    ki_sentences = {}
    sentences_all[k] = to_sentences(sentences_file)
    for sent in sentences_all[k]:       
        if len(sent) > 0 and re.search(regex_ki, sent):
            #print("res",sent, re.search(regex_ki, sent).string)
            split = re.sub(r"\t", "-", sent).split("-")
            ki_sentences[split[0]] = split[1]
    #print(ki_sentences)
    print(f"{k}:", len(ki_sentences))
    ki_data[k] = ki_sentences

with open(r'generated_data/Leipzig_KI_sentences_all.json', 'w') as f:
    json.dump(ki_data, f)
with open(r"generated_data/Leipzig_KI_sentences_all.txt", "w") as f:
        f.write(str(ki_data))
with open(r'generated_data/Leipzig_sentences_all_year_counts.csv', 'w') as f:
    for k, v in sentences_all.items():
        f.write(f"{k}\t{len(v)}\n")

2003_1M: 104
2013_1M: 213
2007_1M: 278
1996_1M: 124
2017_1M: 418
2023_1M: 2414
2006_1M: 195
1997_1M: 118
2016_1M: 361
2022_1M: 422
2002_1M: 153
2012_1M: 425
2009_1M: 404
2019_1M: 636
1998_1M: 95
2024_10K: 35
2008_1M: 266
2018_1M: 503
1999_1M: 159
2005_1M: 143
2015_1M: 361
2021_1M: 420
2001_1M: 175
2025_1M: 3357
2011_1M: 508
2000_1M: 235
2024_1M: 2575
2010_1M: 398
2004_1M: 113
1995_1M: 116
2014_1M: 300
2020_1M: 403


In [6]:
sent_dict = {}
for (key, sentences) in (sentences_all.items()):
    for sent in sentences:
        split = re.sub(r"\t", "-", sent).split("-")
        if (len(split) > 1):
            sent_dict[f"{key}_{split[0]}"] = split[1]

In [7]:
# get KI-terms in "...words.txt" file
words_all = {}
for p in os.listdir(leipzig_main):
    k = p.replace("deu_news_","")
    words_file = os.path.join(leipzig_main, p, f"{p}-words.txt")
    # Convert words file to tab-separated for csv reading
    #Create temp file
    fh, abs_path = mkstemp()
    with fdopen(fh,'w') as new_file:
        with open(words_file) as old_file:
            for line in old_file:
                new_file.write(line.replace(' ',';').replace('\t',';').replace('"','\\"'))
    #Copy the file permissions from the old file to the new file, then remove the old file and
    copymode(words_file, abs_path)
    remove(words_file)
    move(abs_path, words_file)

    # filter ki_terms
    data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')
    words_all[k] = data[1].tolist()

ki_words = list(set([w.lower() for sub in words_all.values() for w in sub if isinstance(w, str) and w.lower() in ki_terms]))
print("percentage of ki_words in ki_terms:", len(ki_words)/len(ki_terms))
pprint(ki_words)


/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_5147/3607378963.py:19: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_5147/3607378963.py:19: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')
/var/folders/0h/lr036jk55nq7b5pmf0kg0qq40000gn/T/ipykernel_5147/3607378963.py:19: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(words_file, sep=";", encoding='utf-8', header=None, on_bad_lines='skip', escapechar='\\')


percentage of ki_words in ki_terms: 0.7647058823529411
['transformer',
 'gpt-j',
 'copilot',
 'stiftungsmodell',
 'llm',
 'llama',
 'chat-gpt',
 'chatgpt',
 'ki',
 'palm',
 'gemini',
 'sprachmodell',
 'ai',
 'lms',
 'ki-system',
 'ki-tool',
 'ki-modell',
 'plms',
 'llms',
 'gpt-2',
 'gpt',
 'ki-anwendung',
 'gpt-3',
 'chatbot',
 'gpt-4',
 'sprachmodelle']


### TF-IDF

In [8]:
# use tf-idf to get keywords from texts
# corpus a list of sentences/documents
# stopwords_l a list of stopwords in the target language if not english
def calc_tf_idf(corpus, stopwords_l= None):
    corpus_cleaned = [remove_punctuation(doc.lower()) for doc in corpus]
    if stopwords_l:
        corpus_cleaned = [remove_stop_words_from_text(doc.split()) for doc in corpus_cleaned]
        vectorizer = TfidfVectorizer(stop_words=stopwords_l, lowercase=True, ngram_range=(1,1))
    else:
        vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1,1))
    # fit tfidf
    tfidf_m = vectorizer.fit_transform(corpus_cleaned) #transform
    #print(tfidf_m.shape)
    feature_array = vectorizer.get_feature_names_out()
    # top (20?) for every doc
    tfidf_sorting = np.argsort(tfidf_m.toarray()).flatten()[::-1] 
    top_n = feature_array[tfidf_sorting].tolist()[:20]
    print(top_n)
    return top_n #array of list(s?)

In [10]:
tfidf_words = {}
for k in ki_data.keys():
    tfidf_words[k] = calc_tf_idf(ki_data[k].values(), stopwords)

with open(f"generated_data/Leipzig_KI_sentences_tfidf_1.txt", "w") as f:
    json.dump(tfidf_words, f)

['63', 'viertelfinale', 'sprung', 'schaffte', 'satzverlust', 'serena', 'williams', 'spielerin', 'titelverteidigerin', 'usa', '75', 'japan', 'sugiyama', 'ai', 'gardens', 'garniert', 'gedacht', 'gearbeitet', 'geben', 'geburtstag']
['grundlegenden', 'nick', 'barber', 'sas', 'vertriebswege', 'partnern', 'accenture', 'deloitte', 'verantwortlich', 'cap', 'institute', 'aufbau', 'neuer', 'zuvor', 'gemini', 'galerie', 'g20gipfel', 'führte', 'gaensheimer', 'überwachung']
['rechts', 'öffentlichen', 'janssen', 'klassische', 'stünde', 'stiftung', 'prüfung', 'wahrscheinlich', 'anstalt', 'gängigste', 'erklärtermaßen', 'formen', 'erscheint', 'gmbh', 'stiftungsmodell', 'geschichten', 'förder', 'geschäftsjahr', 'friedlichen', 'frische']
['ethymologisch', 'schreiben', 'haltbar', 'weidmann', 'klang', 'erbittert', 'deutschlehrer', 'würdig', 'irgendwie', 'kämpfte', 'einfach', 'ai', 'francs', 'federationcupviertelfinale', 'finanziell', 'fredi', 'freiburg', 'film', 'filbinger', 'figaro']
['finanzberichte', 'k

In [11]:
df_tfidf = pd.DataFrame(data=tfidf_words).T
df_tfidf.to_excel('generated_data/Leipzig_KI_sentences_tfidf_1.xlsx')

In [12]:
with open(f"generated_data/Leipzig_KI_sentences_tfidf.txt", "w") as f:
    json.dump(tfidf_words, f)
df_tfidf = pd.DataFrame(data=tfidf_words).T
df_tfidf.to_excel('generated_data/Leipzig_KI_sentences_tfidf.xlsx')

In [13]:
# TODO: tf-idf means per category?? which categories, ai terms?
''' 
vectorizer = TfidfVectorizer(lowercase= True)
means = {}
for i,doc in enumerate(corpora_lex.values()):
    if doc:
        X = vectorizer.fit_transform(doc).todense()
        # calc means for docs
        mean = np.mean(X)
        # to dict
        means[list(corpora_lex.keys())[i]] = mean
        mean_sorted = sorted(means.items(), key = lambda x : x[1], reverse = True)
pprint(mean_sorted)
'''

' \nvectorizer = TfidfVectorizer(lowercase= True)\nmeans = {}\nfor i,doc in enumerate(corpora_lex.values()):\n    if doc:\n        X = vectorizer.fit_transform(doc).todense()\n        # calc means for docs\n        mean = np.mean(X)\n        # to dict\n        means[list(corpora_lex.keys())[i]] = mean\n        mean_sorted = sorted(means.items(), key = lambda x : x[1], reverse = True)\npprint(mean_sorted)\n'

### Compare 24_10K and 24_1M 
(to see if the small set is redundant)

In [6]:
with open(f"generated_data/Leipzig_KI_sentences_all.json", "r") as f:
    sentences_all = json.load(f)
sents_10k = sentences_all["2024_10K"]
sent_1M = sentences_all["2024_1M"]

In [7]:
same_ids = set(sents_10k.keys()).intersection(set(sent_1M.keys()))
pprint(f"same ids in both corpora: {len(same_ids)}")

L10K_df = pd.DataFrame(list(sents_10k.items()), columns=['text_id', 'sentence'])
L1M_df = pd.DataFrame(list(sent_1M.items()), columns=['text_id', 'sentence'])
L_df = pd.concat([L10K_df, L1M_df], ignore_index=True, sort=False)
print(f"Total sentences in 10K corpus: {L10K_df.shape[0]}, in 1M corpus: {L1M_df.shape[0]}")
print(f"Total sentences in both corpora: {L_df.shape[0]}")

duplicated = L_df[L_df.duplicated(subset=['sentence'], keep=False)]
print(duplicated)

'same ids in both corpora: 0'
Total sentences in 10K corpus: 35, in 1M corpus: 2575
Total sentences in both corpora: 2610
   text_id                                           sentence
0       41  aber das was der maturand in dem erwähnten tex...
2      235  allerdings wirft der einsatz der großen sprach...
39    3443  aber das was der maturand in dem erwähnten tex...
85   20541  allerdings wirft der einsatz der großen sprach...


In [8]:
for k in duplicated['text_id']:
    if k in sent_1M.keys():
        print(f"deleting {k} from 1M corpus")
        del sent_1M[k]
    if k in sents_10k.keys():
        print(f"keeping {k} from 10K corpus")
    if k in sents_10k.keys():
        print(f"deleting {k} from 10K corpus")
        del sents_10k[k]
    if k in sent_1M.keys():
        print(f"keeping {k} from 1M corpus")

#overwrite file
sentences_all["2024"] = {**sents_10k, **sent_1M}
sentences_all.pop("2024_10K", None)
sentences_all.pop("2024_1M", None)
print(f"Total sentences in merged corpus: {len(sentences_all['2024'])}")
with open(r'generated_data/Leipzig_KI_sentences_all_counts.csv', 'w') as f:
    for k, v in sentences_all.items():
        f.write(f"{k}\t{len(v)}\n")
#save sentences_all with 10K and 1M merged
with open(r"generated_data/Leipzig_KI_sentences_all.json", "w") as f:
    json.dump(sentences_all, f)

keeping 41 from 10K corpus
deleting 41 from 10K corpus
keeping 235 from 10K corpus
deleting 235 from 10K corpus
deleting 3443 from 1M corpus
deleting 20541 from 1M corpus
Total sentences in merged corpus: 2606


In [9]:
res = 0
for k in sentences_all.keys():
    res_k = len(sentences_all[k])
    res += res_k

print(f"Total ai sentences in corpora after deleting duplicates: {res}")

Total ai sentences in corpora after deleting duplicates: 16423


### BLEU for translations

In [2]:
t_df = pd.read_csv("generated_data/Leipzig_KI_sentences_24_1M_translated.csv")
t_df.head()

,id,text,translation
0,1703,aber auch der ai act spielt eine rolle und die...,but the ai act also plays a role and the high ...
1,2199,aber auch my ai ist über die brille erreichbar\n,but my ai can also be reached via the glasses\n
2,2321,aber auch technologisch gesehen steckt ki noch...,"but from a technological point of view, ki is ..."
3,2968,aber das etikett ki fördert nicht etwa die kau...,but the ki label does not promote the desire t...
4,3443,aber das was der maturand in dem erwähnten tex...,but what the high school graduate did in the t...


In [40]:

def sample_translated_sentences(transl_df, percent=10, random_state=None):
    n = int(np.ceil(len(transl_df) * float(percent) / 100.0))
    samples = transl_df.sample(n=n, random_state=random_state).reset_index(drop=True)
    samples.to_excel(f"generated_data/Leipzig_KI_sentences_24_1M_translated_sample_{percent}percent.xlsx", index=False)
    return samples

sample_df = sample_translated_sentences(t_df, percent=1, random_state=42)
print("length of sample:", len(sample_df))
sample_df.head()

length of sample: 26


,id,text,translation
0,667767,künstliche intelligenz an schulen ist nicht neu\n,artificial intelligence in schools is not new\n
1,503257,es müsse auch darum gehen ki an bestimmten ste...,it must also be about making ki more efficient...
2,201894,das team sieht außerdem das potenzial dass hyb...,the team also sees the potential for hybrid mo...
3,667780,künstliche intelligenz ist an sachsens schulen...,artificial intelligence has arrived at saxony'...
4,272780,der siemens industrial copilot soll ab sommer ...,the siemens industrial Copilot will be availab...


In [49]:
annotator_ref = pd.read_excel("generated_data/Leipzig_KI_sentences_24_1M_translated_sample_1percent-2-2.xlsx")
transl_df = pd.merge(sample_df, annotator_ref, on='id', how='outer')
transl_df = transl_df.dropna(subset=['translation', 'annotator_translation'])

In [ ]:
# calc bleu for samples
# TODO:Define desired weights (example: higher weight for bi-grams)
# read paper for more info on weights
weights = (0.25, 0.25, 0, 0)  # Weights for uni-gram, bi-gram, tri-gram, and 4-gram

# Reference and predicted texts (same as before)
# order sample by id and pick first translation as prediction
predictions = transl_df['translation'].to_list()
annotator_ref = [[row['annotator_translation'], row['translation2']] if pd.notna(row['translation2']) else [row['annotator_translation']] for _, row in transl_df.iterrows()]
print("predictions:", predictions)
print("annotator_ref:", annotator_ref)
# Calculate BLEU score with weights
score = sentence_bleu(annotator_ref, predictions, weights=weights)
# 0 bad, 1 perfect
print("------------------\n"+"BLEU score:", score)

predictions: ["remarkable five created with the help of ki laut niewalda a sign of the genre's willingness to experiment\n", 'the team also sees the potential for hybrid models based on ki and physical models to find their way into other areas, such as materials research\n', 'this can be seen, for example, in the above-average scepticism towards ki in this country\n', 'the online giant amazon enters a new league on the nasdaq - also thanks to the hype surrounding artificial intelligence\n', 'the siemens industrial Copilot will be available for download from summer 2024\n', 'the ability to interact is one of the outstanding achievements with which artificial intelligence has impressed in recent years, according to the ds parent company stellantis\n', 'artificial intelligence ki will set us free\n', 'the mac version of final cut pro can also automatically adjust light and colour using ki, has an optimised slowmotion mode for applesilicon machines only, renameable workflows and an improve